In [1]:
from import_dados import *

In [2]:
sinistros = read_sinistros()

In [ ]:
sinistros = sinistros.with_columns(pl.col('hora_sinistro').str.split_exact(by=':', n=1)
                       .struct.rename_fields(['hora', 'minuto']).alias('hora_struct')
                         ).unnest('hora_struct') \
         .with_columns(pl.col('hora').cast(pl.Int16), 
                       pl.col('minuto').cast(pl.Int16),
                       pl.when(pl.col('tipo_registro') == 'SINISTRO FATAL').then(pl.lit('SIM')).otherwise(pl.lit('NAO')).alias('acidente_fatal'))

In [4]:
sinistros.columns

['id_sinistro',
 'tipo_registro',
 'data_sinistro',
 'ano_sinistro',
 'mes_sinistro',
 'dia_sinistro',
 'hora_sinistro',
 'ano_mes_sinistro',
 'dia_da_semana',
 'turno',
 'logradouro',
 'numero_logradouro',
 'tipo_via',
 'tipo_local',
 'latitude',
 'longitude',
 'cod_ibge',
 'municipio',
 'regiao_administrativa',
 'administracao',
 'conservacao',
 'circunscricao',
 'tp_sinistro_primario',
 'qtd_pedestre',
 'qtd_bicicleta',
 'qtd_motocicleta',
 'qtd_automovel',
 'qtd_onibus',
 'qtd_caminhao',
 'qtd_veic_outros',
 'qtd_veic_nao_disponivel',
 'qtd_gravidade_fatal',
 'qtd_gravidade_grave',
 'qtd_gravidade_leve',
 'qtd_gravidade_ileso',
 'qtd_gravidade_nao_disponivel',
 'tp_sinistro_atrop_pedestre',
 'tp_sinistro_atrop_vitima_fora_veic',
 'tp_sinistro_colisao_frontal',
 'tp_sinistro_colisao_traseira',
 'tp_sinistro_colisao_lateral',
 'tp_sinistro_colisao_transversal',
 'tp_sinistro_colisao_outros',
 'tp_sinistro_choque',
 'tp_sinistro_atrop_animal',
 'tp_sinistro_capotamento',
 'tp_sinistro

In [ ]:
sinistros.filter(pl.col('tipo_via') != ND) \
    .filter(pl.col('turno') != ND) \
    .group_by('dia_da_semana', 'tipo_via', 'acidente_fatal', 'turno').agg(pl.len().alias('acidentes')) \
    .plot.bar(x=alt.X(**get_col_order('turno')), 
              y=alt.Y('acidentes').stack(None), 
              color=alt.Color(**get_col_order('acidente_fatal')), 
              row=alt.Column(**get_col_order('dia_da_semana')), 
              column='tipo_via').properties(height=100, width=300)

alt.Chart(...)

In [15]:
sinistros.filter(pl.col('tipo_via') != 'NAO DISPONIVEL') \
    .filter(pl.col('acidente_fatal') == 'SIM') \
    .filter(pl.col('turno') != ND) \
    .group_by('dia_da_semana', 'tipo_via', 'turno').agg(pl.len().alias('acidentes')) \
    .plot.bar(alt.X(**get_col_order('turno')), 
              y='acidentes:Q', row=alt.Row(**get_col_order('dia_da_semana')), column='tipo_via').properties(height=150, width=300)

alt.Chart(...)

In [16]:
sinistros.filter(pl.col('tipo_via') != 'NAO DISPONIVEL') \
    .filter(pl.col('acidente_fatal') == 'SIM') \
    .group_by('dia_da_semana', 'tipo_via', 'turno').agg(pl.len().alias('acidentes')) \
    .plot.rect(x=alt.X(**get_col_order('turno')), color='acidentes', y=alt.Y(**get_col_order('dia_da_semana')), column='tipo_via').properties(height=200, width=200)


alt.Chart(...)

In [17]:
sinistros.filter(pl.col('tipo_via') != 'NAO DISPONIVEL') \
    .filter(pl.col('turno') != ND) \
    .filter(pl.col('acidente_fatal') == 'SIM') \
    .group_by('dia_da_semana', 'tipo_via', 'turno').agg(pl.sum('qtd_gravidade_fatal').alias('mortes')) \
    .plot.rect(x=alt.X(**get_col_order('turno')), color='mortes', y=alt.Y(**get_col_order('dia_da_semana')), column='tipo_via').properties(height=300, width=250)

alt.Chart(...)

In [18]:
sinistros.filter(pl.col('tipo_via') != 'NAO DISPONIVEL') \
    .filter(pl.col('acidente_fatal') == 'NAO') \
    .group_by('dia_da_semana', 'tipo_via', 'turno').agg(pl.len().alias('mortes')) \
    .plot.rect(x=alt.X(**get_col_order('turno')), color='mortes', y=alt.Y(**get_col_order('dia_da_semana')), column='tipo_via').properties(height=300, width=250)

alt.Chart(...)

In [19]:
sinistros.filter(pl.col('tipo_via') != 'NAO DISPONIVEL') \
    .group_by('dia_da_semana', 'hora', 'tipo_via').agg(pl.len().alias('acidentes')) \
    .with_columns(pl.col('hora').cast(pl.String).str.zfill(2)) \
    .plot.rect(x='hora', color='acidentes', y=alt.Y(**get_col_order('dia_da_semana')), row='tipo_via').properties(height=200, width=550)


alt.Chart(...)

In [20]:
sinistros.filter(pl.col('tipo_via') != 'NAO DISPONIVEL') \
    .filter(pl.col('acidente_fatal') == 'SIM') \
    .group_by('dia_da_semana', 'hora', 'tipo_via').agg(pl.len().alias('acidentes_fatais')) \
    .with_columns(pl.col('hora').cast(pl.String).str.zfill(2)) \
    .plot.rect(x='hora', color='acidentes_fatais', y=alt.Y(**get_col_order('dia_da_semana')), row='tipo_via').properties(height=200, width=550)


alt.Chart(...)